In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize(232),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
val_tf = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

In [ ]:
# ============================================================
# TASK 1 — FROM SCRATCH BASELINE
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import time

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

print(f"Device: {device}")

# ------------------------------------------------------------
# 1. Transforms
# ------------------------------------------------------------
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize(232),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

val_tf = transforms.Compose([
    transforms.Resize(232),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# ------------------------------------------------------------
# 2. Dataset and DataLoader
# ------------------------------------------------------------
train_ds = datasets.Flowers102(
    root="data",
    split="train",
    transform=train_tf,
    download=True
)

val_ds = datasets.Flowers102(
    root="data",
    split="val",
    transform=val_tf,
    download=True
)

test_ds = datasets.Flowers102(
    root="data",
    split="test",
    transform=val_tf,
    download=True
)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

# ------------------------------------------------------------
# 3. CNN Model — Built From Scratch
# ------------------------------------------------------------
class SmallCNN(nn.Module):

    def __init__(self, num_classes=102):
        super().__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),     # 224x224x32
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                                 # 112x112x32

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),    # 112x112x64
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),                                 # 56x56x64

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),   # 56x56x128
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),                                 # 28x28x128

            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),  # 28x28x256
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),                                 # 14x14x256

            # Block 5
            nn.Conv2d(256, 256, kernel_size=3, padding=1),  # 14x14x256
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),                    # 4x4x256
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),             # 4*4*256 = 4096
            nn.Linear(4096, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = SmallCNN(num_classes=102).to(device)

# Number of parameters
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")

# ------------------------------------------------------------
# 4. Loss, Optimizer, Scheduler
# ------------------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=15
)

# ------------------------------------------------------------
# 5. Train and Evaluation Functions
# ------------------------------------------------------------
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for imgs, labels in loader:

        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item() * imgs.size(0)

        correct += (outputs.argmax(1) == labels).sum().item()

        total += imgs.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for imgs, labels in loader:

            imgs = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)

            loss = criterion(outputs, labels)

            total_loss += loss.item() * imgs.size(0)

            correct += (outputs.argmax(1) == labels).sum().item()

            total += imgs.size(0)

    return total_loss / total, correct / total

# ------------------------------------------------------------
# 6. Training Loop — 15 Epochs
# ------------------------------------------------------------
EPOCHS = 15

history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

best_val_acc = 0.0

start_time = time.time()

for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    # Save best model
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_scratch.pth"
        )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.3f}"
    )

total_time = time.time() - start_time

print(f"\nTraining time: {total_time / 60:.1f} minutes")

print(f"Best Validation Accuracy: {best_val_acc:.3f}")

# ------------------------------------------------------------
# 7. Test Accuracy
# ------------------------------------------------------------
model.load_state_dict(torch.load("best_scratch.pth"))

test_loss, test_acc = evaluate(
    model,
    test_loader,
    criterion
)

print(f"Test Accuracy: {test_acc:.3f}")

# ------------------------------------------------------------
# 8. Training Curves
# ------------------------------------------------------------
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(
    epochs_range,
    history["train_loss"],
    label="Train Loss",
    marker="o"
)

axes[0].plot(
    epochs_range,
    history["val_loss"],
    label="Validation Loss",
    marker="o"
)

axes[0].set_title("Task 1 — Loss (From Scratch)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy curve
axes[1].plot(
    epochs_range,
    history["train_acc"],
    label="Train Accuracy",
    marker="o"
)

axes[1].plot(
    epochs_range,
    history["val_acc"],
    label="Validation Accuracy",
    marker="o"
)

axes[1].set_title("Task 1 — Accuracy (From Scratch)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()

plt.savefig(
    "task1_curves.png",
    dpi=150
)

plt.show()

# ------------------------------------------------------------
# Final Results
# ------------------------------------------------------------
print(f"\n{'='*50}")
print("TASK 1 RESULTS")
print(f"{'='*50}")

print(f"Total Parameters:      {total_params:,}")

print(
    f"Best Validation Acc:   "
    f"{best_val_acc:.3f} ({best_val_acc * 100:.1f}%)"
)

print(
    f"Test Accuracy:         "
    f"{test_acc:.3f} ({test_acc * 100:.1f}%)"
)

print(
    f"Training Time:         "
    f"{total_time / 60:.1f} minutes"
)

TASK 1 RESULTS
Total Parameters:      3,129,958
Best Validation Acc:   0.269 (26.9%)
Test Accuracy:         0.209 (20.9%)
Training Time:         3.4 minutes

In [ ]:
 TASK 2 — FEATURE EXTRACTION WITH ResNet18
# ============================================================

from torchvision import models
import time

# ------------------------------------------------------------
# 1. Load Pretrained ResNet18
# ------------------------------------------------------------
model2 = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# ------------------------------------------------------------
# 2. Freeze the Entire Backbone
# ------------------------------------------------------------
for p in model2.parameters():
    p.requires_grad = False

# ------------------------------------------------------------
# 3. Replace the Final Layer (102 Classes)
# ------------------------------------------------------------
model2.fc = nn.Linear(
    model2.fc.in_features,
    102
)

# The new fc layer automatically has requires_grad=True

model2 = model2.to(device)

# ------------------------------------------------------------
# 4. Check Trainable vs Total Parameters
# ------------------------------------------------------------
total_params = sum(
    p.numel() for p in model2.parameters()
)

trainable_params = sum(
    p.numel() for p in model2.parameters()
    if p.requires_grad
)

print(f"Total parameters:      {total_params:,}")

print(f"Trainable parameters:  {trainable_params:,}")

print(f"Frozen parameters:     {total_params - trainable_params:,}")

print(
    f"Trainable percentage:  "
    f"{trainable_params / total_params * 100:.2f}%"
)

# Show which layers are trainable
print("\nTrainable layers:")

for name, param in model2.named_parameters():

    if param.requires_grad:
        print(f"  ✅ {name}")

# ------------------------------------------------------------
# 5. Optimizer — Only Trainable Parameters
# ------------------------------------------------------------
trainable = filter(
    lambda p: p.requires_grad,
    model2.parameters()
)

optimizer2 = optim.Adam(
    trainable,
    lr=1e-3
)

scheduler2 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer2,
    T_max=15
)

criterion = nn.CrossEntropyLoss()

# ------------------------------------------------------------
# 6. Training Loop — 15 Epochs
# ------------------------------------------------------------
EPOCHS = 15

history2 = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

best_val_acc2 = 0.0

start_time2 = time.time()

for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc = train_one_epoch(
        model2,
        train_loader,
        optimizer2,
        criterion
    )

    val_loss, val_acc = evaluate(
        model2,
        val_loader,
        criterion
    )

    scheduler2.step()

    history2["train_loss"].append(train_loss)
    history2["val_loss"].append(val_loss)
    history2["train_acc"].append(train_acc)
    history2["val_acc"].append(val_acc)

    # Save best model
    if val_acc > best_val_acc2:

        best_val_acc2 = val_acc

        torch.save(
            model2.state_dict(),
            "best_feature_extraction.pth"
        )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.3f}"
    )

total_time2 = time.time() - start_time2

print(f"\nTraining time: {total_time2 / 60:.1f} minutes")

print(f"Best Validation Accuracy: {best_val_acc2:.3f}")

# ------------------------------------------------------------
# 7. Test Accuracy
# ------------------------------------------------------------
model2.load_state_dict(
    torch.load(
        "best_feature_extraction.pth",
        weights_only=True
    )
)

test_loss2, test_acc2 = evaluate(
    model2,
    test_loader,
    criterion
)

print(f"Test Accuracy: {test_acc2:.3f}")

# ------------------------------------------------------------
# 8. Training Curves
# ------------------------------------------------------------
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(
    epochs_range,
    history2["train_loss"],
    label="Train Loss",
    marker="o"
)

axes[0].plot(
    epochs_range,
    history2["val_loss"],
    label="Validation Loss",
    marker="o"
)

axes[0].set_title("Task 2 — Loss (Feature Extraction)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy curve
axes[1].plot(
    epochs_range,
    history2["train_acc"],
    label="Train Accuracy",
    marker="o"
)

axes[1].plot(
    epochs_range,
    history2["val_acc"],
    label="Validation Accuracy",
    marker="o"
)

axes[1].set_title("Task 2 — Accuracy (Feature Extraction)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()

plt.savefig(
    "task2_curves.png",
    dpi=150
)

plt.show()

# ------------------------------------------------------------
# 9. Final Results
# ------------------------------------------------------------
print(f"\n{'=' * 50}")

print("TASK 2 RESULTS")

print(f"{'=' * 50}")

print(f"Total parameters:      {total_params:,}")

print(f"Trainable parameters:  {trainable_params:,}")

print(
    f"Best Validation Acc:   "
    f"{best_val_acc2:.3f} ({best_val_acc2 * 100:.1f}%)"
)

print(
    f"Test Accuracy:         "
    f"{test_acc2:.3f} ({test_acc2 * 100:.1f}%)"
)

print(
    f"Training time:         "
    f"{total_time2 / 60:.1f} minutes"
)

TASK 2 RESULTS
Total parameters:      11,228,838
Trainable parameters:  52,326
Best Validation Acc:   0.842 (84.2%)
Test Accuracy:         0.822 (82.2%)
Training time:         3.1 minutes

In [ ]:
import copy

# ------------------------------------------------------------
# 1. Load the Best Model from Task 2
# ------------------------------------------------------------
# Load the best feature extraction model
model3 = models.resnet18(weights=None)

model3.fc = nn.Linear(
    model3.fc.in_features,
    102
)

model3.load_state_dict(
    torch.load(
        "best_feature_extraction.pth",
        weights_only=True
    )
)

model3 = model3.to(device)

print("Task 2 model loaded successfully ✅")

# ------------------------------------------------------------
# 2. Unfreeze layer4 and fc, Freeze Everything Else
# ------------------------------------------------------------

# First freeze all layers
for p in model3.parameters():
    p.requires_grad = False

# Unfreeze layer4
for p in model3.layer4.parameters():
    p.requires_grad = True

# Unfreeze final classifier
for p in model3.fc.parameters():
    p.requires_grad = True

# Check trainable parameters
total_params = sum(
    p.numel() for p in model3.parameters()
)

trainable_params = sum(
    p.numel() for p in model3.parameters()
    if p.requires_grad
)

print(f"\nTotal parameters:      {total_params:,}")

print(f"Trainable parameters:  {trainable_params:,}")

print(f"Frozen parameters:     {total_params - trainable_params:,}")

# Show trainable layers
print("\nTrainable layers:")

for name, param in model3.named_parameters():

    if param.requires_grad:
        print(f"  ✅ {name}")

# ------------------------------------------------------------
# 3. Optimizer with Discriminative Learning Rates
# ------------------------------------------------------------
optimizer3 = optim.Adam([

    # Smaller learning rate for pretrained backbone
    {
        "params": model3.layer4.parameters(),
        "lr": 1e-5
    },

    # Larger learning rate for classifier
    {
        "params": model3.fc.parameters(),
        "lr": 1e-3
    },

])

scheduler3 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer3,
    T_max=10
)

criterion = nn.CrossEntropyLoss()

# ------------------------------------------------------------
# 4. Training Loop — 10 Epochs
# ------------------------------------------------------------
EPOCHS = 10

history3 = {
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": []
}

best_val_acc3 = 0.0

start_time3 = time.time()

for epoch in range(1, EPOCHS + 1):

    train_loss, train_acc = train_one_epoch(
        model3,
        train_loader,
        optimizer3,
        criterion
    )

    val_loss, val_acc = evaluate(
        model3,
        val_loader,
        criterion
    )

    scheduler3.step()

    history3["train_loss"].append(train_loss)
    history3["val_loss"].append(val_loss)
    history3["train_acc"].append(train_acc)
    history3["val_acc"].append(val_acc)

    # Save best model
    if val_acc > best_val_acc3:

        best_val_acc3 = val_acc

        torch.save(
            model3.state_dict(),
            "best_fine_tuned.pth"
        )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.3f}"
    )

total_time3 = time.time() - start_time3

print(f"\nTraining time: {total_time3 / 60:.1f} minutes")

print(f"Best Validation Accuracy: {best_val_acc3:.3f}")

# ------------------------------------------------------------
# 5. Test Accuracy
# ------------------------------------------------------------
model3.load_state_dict(
    torch.load(
        "best_fine_tuned.pth",
        weights_only=True
    )
)

test_loss3, test_acc3 = evaluate(
    model3,
    test_loader,
    criterion
)

print(f"Test Accuracy: {test_acc3:.3f}")

# ------------------------------------------------------------
# 6. Training Curves
# ------------------------------------------------------------
epochs_range = range(1, EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(
    epochs_range,
    history3["train_loss"],
    label="Train Loss",
    marker="o"
)

axes[0].plot(
    epochs_range,
    history3["val_loss"],
    label="Validation Loss",
    marker="o"
)

axes[0].set_title("Task 3 — Loss (Fine-Tuning)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy curve
axes[1].plot(
    epochs_range,
    history3["train_acc"],
    label="Train Accuracy",
    marker="o"
)

axes[1].plot(
    epochs_range,
    history3["val_acc"],
    label="Validation Accuracy",
    marker="o"
)

axes[1].set_title("Task 3 — Accuracy (Fine-Tuning)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()

plt.savefig(
    "task3_curves.png",
    dpi=150
)

plt.show()

# ------------------------------------------------------------
# 7. Comparison of the Three Approaches
# ------------------------------------------------------------
print(f"""
{'=' * 65}
COMPARISON TABLE
{'=' * 65}
{'Approach':<25} {'Train.Param':>12} {'Val Acc':>10} {'Test Acc':>10} {'Time':>8}
{'-' * 65}
{'From Scratch (Task 1)':<25} {'3,129,958':>12} {'26.9%':>10} {'20.9%':>10} {'3.4 min':>8}
{'Feature Ext (Task 2)':<25} {'52,326':>12} {'84.2%':>10} {'82.2%':>10} {'3.1 min':>8}
{'Fine-Tune (Task 3)':<25} {f'{trainable_params:,}':>12} {f'{best_val_acc3*100:.1f}%':>10} {f'{test_acc3*100:.1f}%':>10} {f'{total_time3/60:.1f} min':>8}
{'=' * 65}
""")

# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------
summary = f"""
SUMMARY (4–6 sentences):
{'=' * 65}

Transfer learning created a dramatic improvement compared to
training from scratch. With feature extraction, validation
accuracy increased from 26.9% to 84.2% — more than a 3x
improvement. Fine-tuning (Task 3) further improved performance
by unfreezing layer4 and reached {best_val_acc3*100:.1f}%
validation accuracy — {(best_val_acc3 - 0.842)*100:.1f}%
better than pure feature extraction.

For small datasets, feature extraction is usually the safest
and fastest approach because it provides strong performance
while minimizing overfitting risk.

{'=' * 65}
"""

print(summary)

# ------------------------------------------------------------
# 9. Comparison Plot for All Tasks
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))

# Task 1
ax.plot(
    range(1, 16),
    history["val_acc"],
    label="Task 1 — From Scratch",
    marker="o",
    linestyle="--"
)

# Task 2
ax.plot(
    range(1, 16),
    history2["val_acc"],
    label="Task 2 — Feature Extraction",
    marker="s"
)

# Task 3
ax.plot(
    range(16, 26),
    history3["val_acc"],
    label="Task 3 — Fine-Tuning",
    marker="^"
)

ax.axvline(
    x=15,
    color="gray",
    linestyle=":",
    label="Fine-Tuning Starts"
)

ax.set_title("All Tasks — Validation Accuracy Comparison")

ax.set_xlabel("Epoch")

ax.set_ylabel("Validation Accuracy")

ax.legend()

ax.grid(True)

plt.tight_layout()

plt.savefig(
    "all_tasks_comparison.png",
    dpi=150
)

plt.show()

COMPARISON TABLE
Approach                   Train.Param    Val Acc   Test Acc     Time
From Scratch (Task 1)        3,129,958      26.9%      20.9%  3.4 min
Feature Ext (Task 2)            52,326      84.2%      82.2%  3.1 min
Fine-Tune (Task 3)           8,446,054      87.4%      85.8%  2.1 min

Transfer learning led to a significant performance boost compared to training the model from scratch. Using feature extraction, validation accuracy improved from 26.9% to 84.2%, showing a strong gain in performance. Further fine-tuning by unfreezing layer4 increased the accuracy to 87.4%, adding an additional 3.2% improvement over feature extraction alone. These results show that fine-tuning can push performance even higher when carefully applied. Overall, for small datasets, feature extraction remains the most stable and efficient approach, while fine-tuning offers extra gains when additional accuracy is needed.
